## 0️⃣ Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import model as M   # our ML pipeline
print('Imports successful')

## 1️⃣ Load & Explore Data

In [ ]:
df_raw = M.load_data('aqi.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('=== Data Types ===')
print(df_raw.dtypes)
print()
print('=== Null Counts ===')
print(df_raw.isnull().sum())

In [ ]:
df_raw.describe()

In [ ]:
print('=== Air Quality Status Distribution ===')
print(df_raw['air_quality_status'].value_counts())

In [ ]:
fig = M.plot_aqi_distribution(df_raw)
plt.show()

## 2️⃣ Preprocess

In [ ]:
df = M.preprocess(df_raw)
print(f'Shape after preprocessing: {df.shape}')
print(f'New columns: {[c for c in df.columns if c not in df_raw.columns]}')
df.head()

## 3️⃣ Feature Engineering

In [ ]:
df = M.engineer_features(df)
print('New features added:')
print(df[['prominent_pollutants', 'primary_pollutant', 'multi_pollutant', 'aqi_value', 'aqi_bucket']].head(10))

## 4️⃣ Encode Categoricals

In [ ]:
df_enc = M.encode_features(df, fit=True)
print('Encoded sample:')
df_enc[M.CATEGORICAL_COLS].head()

## 5️⃣ Train Model

In [ ]:
print('Training RandomForestClassifier ...')
model, X_test, y_test, le_target = M.train(df_enc)
print('Training complete!')
print(f'Model saved to: {M.MODEL_FILE}')

## 6️⃣ Evaluate

In [ ]:
results = M.evaluate(model, X_test, y_test, le_target)
print(f'Accuracy: {results["accuracy"]*100:.2f}%')

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(
    y_test,
    model.predict(X_test),
    target_names=le_target.classes_
))

In [ ]:
fig = M.plot_confusion_matrix(results['cm'], results['classes'])
plt.show()

In [ ]:
fig = M.plot_feature_importance(model, M.FEATURE_COLS)
plt.show()

## 7️⃣ Predict on New Samples

In [ ]:
samples = [
    dict(state='Delhi',       area='Anand Vihar',        aqi_value=380, prominent_pollutants='PM2.5,PM10', num_stations=5, month=11, year=2024),
    dict(state='Maharashtra', area='Pune',               aqi_value=160, prominent_pollutants='PM2.5',      num_stations=3, month=3,  year=2024),
    dict(state='Kerala',      area='Thiruvananthapuram', aqi_value=42,  prominent_pollutants='O3',          num_stations=2, month=6,  year=2024),
    dict(state='Rajasthan',   area='Jodhpur',            aqi_value=290, prominent_pollutants='PM10,CO',    num_stations=2, month=1,  year=2024),
]

for s in samples:
    result = M.predict(**s)
    print(f"{s['state']:15s} | AQI {s['aqi_value']:3d} | {s['prominent_pollutants']:15s} → "
          f"[{result['label']:12s}]  Confidence: {result['confidence']}%")